<img src="https://raw.githubusercontent.com/aai-services/datainsight-curriculum/main/assets/brand/banner-stage-1.png" alt="Data Insight, Stage 1: Foundations" width="100%">

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aai-services/datainsight-curriculum/blob/main/stages/stage-1/1.2-tables-with-pandas/lesson.ipynb)

**Lesson 1.2**

# Tables with pandas

Stage 1 · Foundations  |  Week 4  |  About 10 hours with practice  |  AI use: tutor only

### Why this matters

A researcher hands you a table of 344 penguins. Within five minutes you should be able to say how big it is, what it contains, what is missing, and answer simple questions such as which island has the heaviest birds. That first look at a new table is the skill this lesson builds, and it is the first step of every analysis you will ever do.

### Learning objectives

By the end of this lesson you will be able to:

1. Load a CSV file into a pandas DataFrame and inspect its size, columns, types, and gaps
2. Select columns and rows, and filter rows by one or more conditions
3. Sort, count, and add calculated columns

**Data:** Palmer Penguins: measurements of 344 penguins of three species in Antarctica, 2007 to 2009 (Horst, Hill and Gorman, 2020; CC0).

### Warm-up

Answer these from memory before looking. Recalling what you learned, rather than rereading it, is one of the most effective ways to make it stick.

1. What does `len([4, 8, 15])` return?

<details>
<summary>Show answer</summary>

`3`, the number of items in the list.

</details>

2. What is the difference between a list and a dictionary?

<details>
<summary>Show answer</summary>

A list keeps values in order by position; a dictionary looks values up by key.

</details>

> **How to use this notebook.** Run each cell in order with Shift+Enter. When you see **Predict**, write down your answer before running the next cell; being wrong and then seeing why is how the ideas stick. **Try it** exercises have a hidden solution: attempt them first. None of this is graded.

In [ ]:
import pandas as pd

# Where the course datasets are stored
DATA = "https://raw.githubusercontent.com/aai-services/datainsight-curriculum/main/datasets/"

penguins = pd.read_csv(DATA + "penguins.csv")
penguins.head()

## 1. First look at any new table

Always start with these four questions: how big is it, what are the columns, what types are they, and what is missing?

> **Predict.** From the first rows you have seen, how many columns do you expect? Guess the number of rows too.

In [ ]:
print(penguins.shape)          # (rows, columns)
print(penguins.columns.tolist())

In [ ]:
penguins.info()                # types and non-missing counts per column

In [ ]:
penguins.describe()            # summary of the numeric columns

**What to notice:** `body_mass_g` has 342 non-missing values out of 344, so two penguins were not weighed. The `sex` column has more gaps. Missing values appear as `NaN` and are left out of summaries like the mean.

## 2. Selecting columns

One column gives a **Series**; a list of columns gives a smaller **DataFrame**.

In [ ]:
penguins["species"].head()

In [ ]:
penguins[["species", "island", "body_mass_g"]].head()

> **Check your understanding.** What is the difference between `penguins["species"]` and `penguins[["species"]]`?

<details>
<summary>Show answer</summary>

The first returns a **Series** (one column). The second, with a list of column names, returns a **DataFrame** with one column.

</details>

## 3. Filtering rows

A condition produces `True` or `False` for every row; using it inside `[ ]` keeps the `True` rows. Combine conditions with `&` (and) or `|` (or), with each condition in parentheses.

> **Predict.** Will more or fewer than half of the penguins weigh over 5 kg?

In [ ]:
heavy = penguins[penguins["body_mass_g"] > 5000]
print(len(heavy), "penguins weigh more than 5 kg")

heavy_on_biscoe = penguins[(penguins["body_mass_g"] > 5000) & (penguins["island"] == "Biscoe")]
len(heavy_on_biscoe)

`.loc` selects rows and columns together, by condition or by label:

In [ ]:
penguins.loc[penguins["species"] == "Chinstrap", ["island", "flipper_length_mm"]].head()

> **Try it.** How many Adelie penguins were measured on Dream island?

In [ ]:
# Keep rows where species is "Adelie" and island is "Dream", then count them.
...

<details>
<summary>Show a solution</summary>

```python
adelie_dream = penguins[(penguins["species"] == "Adelie") & (penguins["island"] == "Dream")]
len(adelie_dream)   # 56
```

</details>

> **Common mistake.** `penguins[penguins["island"] = "Dream"]` uses a single `=`, which assigns a value. Comparison needs `==`.

> **Check your understanding.** Why does each condition need its own parentheses when combined with `&`?

<details>
<summary>Show answer</summary>

In Python, `&` is applied before comparisons such as `>` and `==`. Without parentheses, pandas tries to combine the wrong pieces and raises an error.

</details>

## 4. Sorting and counting

In [ ]:
penguins.sort_values("body_mass_g", ascending=False).head(3)

> **Predict.** Which island do you expect to have the most penguins?

In [ ]:
penguins["island"].value_counts()

> **Common mistake.** `sort_values` returns a **new** table. `penguins` itself does not change unless you assign the result back to a name.

## 5. New columns

Operations on a column apply to every row at once, with no loop needed.

In [ ]:
penguins["body_mass_kg"] = penguins["body_mass_g"] / 1000
penguins[["body_mass_g", "body_mass_kg"]].head()

> **Try it.** Which penguin has the deepest bill? Show its species, island, and bill depth.

In [ ]:
# Sort by bill depth, deepest first, and show one row with three columns.
...

<details>
<summary>Show a solution</summary>

```python
penguins.sort_values("bill_depth_mm", ascending=False).head(1)[["species", "island", "bill_depth_mm"]]
# An Adelie penguin on Torgersen, 21.5 mm
```

</details>

## 6. Worked example: which species has the longest flippers relative to its weight?

A ratio lets us compare penguins of different sizes. We compute flipper length per kilogram, then compare species.

In [ ]:
penguins["flipper_per_kg"] = penguins["flipper_length_mm"] / penguins["body_mass_kg"]

for species in ["Adelie", "Chinstrap", "Gentoo"]:
    rows = penguins[penguins["species"] == species]
    print(f"{species:10s} median {rows['flipper_per_kg'].median():.1f} mm per kg  (n = {rows['flipper_per_kg'].notna().sum()})")

**Check the result.** The number of penguins used for each species should add up to the number with both measurements.

In [ ]:
both = penguins[["flipper_length_mm", "body_mass_g"]].notna().all(axis=1).sum()
assert both == penguins["flipper_per_kg"].notna().sum()
print(both, "penguins have both measurements")

**Interpretation.** Chinstrap and Adelie penguins have longer flippers for their weight than Gentoo penguins, which are heavier. This describes these 344 penguins; whether the difference is reliable enough to generalise is the question of lesson 1.6.

In the next lesson you will learn a shorter way to get results per group, with `groupby`.

## Reflect

Before moving on, write two sentences in your own notes: the most important idea from this lesson, and the part that is still least clear. Bring the second one to your peer group or the office hour.

## Key takeaways

- Start every analysis by checking shape, columns, types, and missing values.
- A condition inside `[ ]` keeps the rows where it is `True`.
- Operations on a column apply to every row at once; no loop is needed.
- Summaries such as the mean skip missing values, so always report how many values were used.

## Next step

Open `practice-1.2.ipynb` in your Stage 1 practice repository and complete it until its automatic check passes.

**Apply it to your own context.** Find a CSV file published by your national statistics office and answer the four first-look questions: how big is it, what are the columns, what types are they, and what is missing?

---

<img src="https://raw.githubusercontent.com/aai-services/datainsight-curriculum/main/assets/brand/mark.png" alt="Data Insight" width="32">

**Data Insight** · Open data science curriculum · [datainsightonline.com](https://datainsightonline.com)

Released under the Creative Commons Attribution 4.0 license (CC BY 4.0).

Suggested citation: Data Insight (2026). *Lesson 1.2: Tables with pandas*. https://github.com/aai-services/datainsight-curriculum